# 07 - Evaluate a fine-tuned model on MATH-500
One notebook for all evaluations. Change `variant` (which model) and `repetition_penalty`.
First round used `repetition_penalty=1.0` (results in `experiments/07_eval`), final round used `1.1` (results in `experiments/08_eval_rp`).
Run on Kaggle (2x T4). All tables are built in notebook 08.

In [ ]:
!pip uninstall -y torchaudio # newer Kaggle image: old torchaudio breaks "import datasets" after vllm upgrades torch
!pip uninstall -y torchao
!pip install -U transformers==5.18.0 peft==0.21.1 vllm math-verify datasets

In [ ]:
import sys
import torch
import transformers
import peft
import vllm

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("vLLM:", vllm.__version__)
print("GPUs:", torch.cuda.device_count())

In [ ]:
variant='cot'          # 'cot' | 'review' | 'refine'
repetition_penalty=1.0 # 1.0 = off, 1.1 = discourage repeating loops

In [ ]:
import glob
import os

adapter_path=os.path.dirname(glob.glob('/kaggle/input/**/*-final/adapter_config.json',recursive=True)[0])
print(adapter_path)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_name='Qwen/Qwen3.5-2B-Base' # 4B run: 'Qwen/Qwen3.5-4B-Base'

tokenizer=AutoTokenizer.from_pretrained(model_name)

base_model=AutoModelForCausalLM.from_pretrained(model_name,dtype=torch.float16)
model=PeftModel.from_pretrained(base_model,adapter_path)
model=model.merge_and_unload()

merged_path=f'/kaggle/working/merged-{variant}'
model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

print("Merged model saved to:", merged_path)

In [ ]:
del model, base_model
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from datasets import load_dataset

dataset=load_dataset("HuggingFaceH4/MATH-500",split="test")
print(dataset)

In [ ]:
def format_prompt(record: dict)->str:
    return '[Question] '+record.get('problem')+'\n'

prompts=[format_prompt(i) for i in dataset]
print(prompts[0])

In [ ]:
from vllm import LLM, SamplingParams

llm=LLM(
    model=merged_path,
    dtype="float16",
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    enforce_eager=True, # skip CUDA graphs: they need extra GPU memory and the 4B model ran out of memory without this
)

print("Model loaded")

In [ ]:
greedy_params=SamplingParams(
    n=1,
    temperature=0.0,
    max_tokens=3072,
    repetition_penalty=repetition_penalty,
)

outputs=llm.generate(prompts,greedy_params)
print("Generation done")

In [ ]:
def extract_boxed(text: str):
    start=text.rfind('\\boxed{')
    if start==-1:
        return None

    i=start+len('\\boxed{')
    depth=1
    while i<len(text) and depth>0:
        if text[i]=='{':
            depth+=1
        elif text[i]=='}':
            depth-=1
        i+=1

    if depth!=0:
        return None
    return text[start+len('\\boxed{'):i-1]

In [ ]:
from math_verify import parse, verify

def is_correct(answer, gold: str)->bool:
    if answer is None:
        return False
    try:
        return bool(verify(parse('$'+gold+'$'),parse('\\boxed{'+answer+'}')))
    except Exception:
        return False

In [ ]:
def get_first_answer(text: str):
    if '[Answer 1]' not in text:
        return None
    part=text.split('[Answer 1]')[1].split('[Answer 2]')[0].split('[Final answer]')[0]
    return extract_boxed(part)

def get_final_answer(text: str):
    if '[Final answer]' not in text:
        return None
    return text.split('[Final answer]')[-1].strip()

In [ ]:
results=[]

for record,output in zip(dataset,outputs):
    text=output.outputs[0].text
    first=get_first_answer(text)
    final=get_final_answer(text)

    results.append({
        'id': record['unique_id'],
        'level': record['level'],
        'gold': record['answer'],
        'first_answer': first,
        'final_answer': final,
        'first_ok': is_correct(first,record['answer']),
        'final_ok': is_correct(final,record['answer']),
        'finished': output.outputs[0].finish_reason=='stop',
        'n_tokens': len(output.outputs[0].token_ids),
        'text': text,
    })

print(len(results))

In [ ]:
n=len(results)

print("Variant:", variant, "| repetition_penalty:", repetition_penalty)
print("Final accuracy:", sum(r['final_ok'] for r in results)/n)
print("First-answer accuracy:", sum(r['first_ok'] for r in results)/n)
readable=[r for r in results if r['first_answer']!=None]
print("First answer readable:", len(readable)/n)
print("Wrong -> right (readable only):", sum((not r['first_ok']) and r['final_ok'] for r in readable)/n)
print("Right -> wrong (readable only):", sum(r['first_ok'] and (not r['final_ok']) for r in readable)/n)
print("Finished (stopped by itself):", sum(r['finished'] for r in results)/n)
print("Avg tokens:", sum(r['n_tokens'] for r in results)/n)

In [ ]:
for level in sorted(set(r['level'] for r in results)):
    subset=[r for r in results if r['level']==level]
    print('Level',level,
          'n=',len(subset),
          'final=',round(sum(r['final_ok'] for r in subset)/len(subset),3),
          'first=',round(sum(r['first_ok'] for r in subset)/len(subset),3))

In [ ]:
print(results[0]['text'][:3000])

In [ ]:
import json
from pathlib import Path

savepath=Path(f'/kaggle/working/eval_{variant}_rp{repetition_penalty}.json')

with savepath.open('w',encoding='utf-8') as file:
    json.dump(results,file,ensure_ascii=False,indent=2)

print("Saved:",savepath)